# Guard a local mock action

Use this folder and the shared root kernel. Proposed actions are AI-authored fixtures. The separate application contexts stand for trusted authenticated server state. A surrounding fixture session selects a test context; the proposal itself never selects its permissions.

In [ ]:
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/proposals.json").read_text())
application_data = json.loads(Path("docs/dataset/application_contexts.json").read_text())
contexts = application_data["contexts"]
pprint(application_data)
pprint(fixtures)

## Read validation and the mock action before importing

`tools/execute.py` checks the tool, exact argument keys/types/range, authenticated employee identity, then application permissions. `tools/mock_actions.py` appends a local record only after allow. Top-level permission/role claims in a proposal are ignored.

In [ ]:
from tools.execute import check_proposal, execute_proposal
pprint(check_proposal(fixtures[0]["proposal"], contexts["employee_session"]))

## Inspect real in-memory action mutations

With max_days=5, only the normal proposal should execute. Each reported count is derived from the action-log length around the actual call; inspect the log itself too.

In [ ]:
baseline_log = []
baseline = {}
for item in fixtures:
    baseline[item["id"]] = execute_proposal(item["proposal"], contexts[item["session"]], baseline_log, max_days=5)
for name, result in baseline.items():
    pprint({"id": name, **result})
pprint({"actual_action_log": baseline_log})

## Model claims cannot authorize execution

The unauthorized proposal has valid arguments and claims permission/admin role, but its app-owned viewer context grants neither. It is blocked before the mock action. The wrong employee proposal is also rejected.

In [ ]:
pprint(fixtures[3])
pprint(baseline["unauthorized_claim"])
assert baseline["unauthorized_claim"]["actual_mock_executions"] == 0
assert baseline["wrong_employee"]["actual_mock_executions"] == 0
assert baseline_log == [{"tool": "request_leave", "employee_id": "EMP-001", "days": 2}]

## Change only max_days and replay identical proposals

Raise max_days from 5 to 10. The same six-day request becomes admissible; permissions and identity rules stay fixed. This cap is an application rule, not leave-balance validation.

In [ ]:
candidate_log = []
candidate = {}
for item in fixtures:
    candidate[item["id"]] = execute_proposal(item["proposal"], contexts[item["session"]], candidate_log, max_days=10)
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "baseline": baseline[name]["decision"], "candidate": candidate[name]["decision"], "reason": candidate[name]["reason"], "executions": candidate[name]["actual_mock_executions"]})
pprint({"actual_candidate_action_log": candidate_log})

## Check execution evidence and explain the limit

Only the normal and six-day proposals executed under the larger cap. This mock has no real HR effect and no API calls. A valid structure alone never grants application authority.

In [ ]:
assert candidate["range_edge"]["decision"] == "allow"
assert candidate["unauthorized_claim"]["decision"] == "block"
assert len(baseline_log) == 1 and len(candidate_log) == 2
assert all(result["actual_mock_executions"] == (1 if result["decision"] == "allow" else 0) for result in candidate.values())
pprint({"baseline_actual_executions": len(baseline_log), "candidate_actual_executions": len(candidate_log), "external_actions": 0})